In [1]:
!pip install -q \
chromadb \
wikipedia-api \
langchain-text-splitters \
google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [2]:

import wikipediaapi
import os
try:
    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="networking-rag-project"
    )

    topics = [
        "Machine learning",
    "Neural network",
    "Deep learning",
    "Transformer model",
    "Natural language processing"
    ]

    os.makedirs("networking_docs", exist_ok=True)

    for topic in topics:

        page = wiki.page(topic)

        if page.exists():

            filename = (
                topic.replace("/", "_")
                     .replace(" ", "_")
            ) + ".txt"

            with open(
                f"networking_docs/{filename}",
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page.text)

            print(f"Saved: {filename}")

        else:
            print(f"Page not found: {topic}")

    print("\nAll documents downloaded successfully!")

except Exception as e:
    print("Download Error:")
    print(e)

Saved: Machine_learning.txt
Saved: Neural_network.txt
Saved: Deep_learning.txt
Saved: Transformer_model.txt
Saved: Natural_language_processing.txt

All documents downloaded successfully!


In [3]:

import os
try:
    documents = []

    folder_path = "networking_docs"

    for file in os.listdir(folder_path):

        if file.endswith(".txt"):

            file_path = os.path.join(folder_path, file)

            with open(file_path, "r", encoding="utf-8") as f:
                documents.append(f.read())

    print("Documents loaded successfully!")
    print("Number of documents:", len(documents))

except Exception as e:
    print("Loading Error:")
    print(e)

Documents loaded successfully!
Number of documents: 5


In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=50
    )

    chunks = []

    for document in documents:

        document_chunks = splitter.split_text(document)

        chunks.extend(document_chunks)

    print("Chunking completed successfully!")
    print("Total Chunks:", len(chunks))

except Exception as e:
    print("Chunking Error:")
    print(e)

Chunking completed successfully!
Total Chunks: 247


In [ ]:
from google import genai
from google.colab import userdata
import time

client = genai.Client(api_key=userdata.get("GEMINI_KEY"))
try:
    chunk_embeddings = []

    for i, chunk in enumerate(chunks):

        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=chunk
        )

        chunk_embeddings.append(
            response.embeddings[0].values
        )

        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1} chunks")

        time.sleep(1)

    print("Embedding generation completed!")
    print("Total embeddings:", len(chunk_embeddings))

except Exception as e:
    print("Embedding Error:")
    print(e)

Processed 20 chunks
Processed 40 chunks
Processed 60 chunks
Processed 80 chunks
Processed 100 chunks
Processed 120 chunks
Processed 140 chunks
Processed 160 chunks
Processed 180 chunks
Processed 200 chunks
Processed 220 chunks
Processed 240 chunks
Embedding generation completed!
Total embeddings: 247


In [ ]:
import chromadb
try:
    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Chunks and embeddings stored successfully!")
    print("Total records:", collection.count())

except Exception as e:
    print("ChromaDB Error:")
    print(e)

Chunks and embeddings stored successfully!
Total records: 247


In [ ]:
import pickle

try:
    with open("chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)

    with open("chunk_embeddings.pkl", "wb") as f:
        pickle.dump(chunk_embeddings, f)

    print("✅ Files saved successfully!")

except Exception as e:
    print("❌ Error saving files:")
    print(e)

✅ Files saved successfully!


In [ ]:
import os
print(os.listdir())

['.config', 'networking_docs', 'networking_chromadb', 'chunk_embeddings.pkl', 'chunks.pkl', 'sample_data']
